# NEXUS — Qwen3.8-Flash-Next on vLLM-TPU (Kaggle TPU v5e-8)

Qualifies the **vLLM → tpu-inference → NEXUS** integration. Not a standalone runtime.

- Attach this repository and an optional pre-converted checkpoint dataset.
- Select **TPU v5e-8** accelerator.
- Set `NEXUS_CHECKPOINT_ROOT` if using a local quantized checkpoint.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

REPO = Path('/kaggle/input/nexus-repo') if Path('/kaggle/input/nexus-repo').exists() else Path('.')
RESULTS = Path('nexus_benchmark_results')
RESULTS.mkdir(exist_ok=True)

def run(cmd):
    print('>>', ' '.join(cmd))
    subprocess.check_call(cmd)

if not (REPO / 'pyproject.toml').exists():
    raise RuntimeError(f'NEXUS repo not found at {REPO}')

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO) + '[dev]'])
if os.environ.get('NEXUS_INSTALL_UPSTREAM', '1') == '1':
    run(['bash', str(REPO / 'scripts/install_upstream.sh')])

In [ ]:
import jax
devices = jax.devices()
tpu_devices = [d for d in devices if 'tpu' in str(d.platform).lower()]
env_report = {
    'jax_version': jax.__version__,
    'device_count': len(devices),
    'tpu_device_count': len(tpu_devices),
    'devices': [str(d) for d in devices],
    'production_ok': len(tpu_devices) == 8,
}
print(json.dumps(env_report, indent=2))
if env_report['production_ok']:
    print('TPU v5e-8 detected (8 chips).')
else:
    print('WARNING: expected 8 TPU devices for production qualification.')

In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pytest', str(REPO / 'tests'), '-q'])
print('CPU smoke tests passed.')

In [ ]:
from nexus.integration.register import register_qwen4_exp
from nexus.config import FlashNextConfig, RuntimeConfig
from nexus.memory.planner import estimate_runtime_memory, live_device_memory
from nexus.quantization.packing import estimate_storage_gib

register_qwen4_exp()

for ngram_bits in (4, 3, 2, 1):
    est = estimate_storage_gib(ngram_bits=ngram_bits)
    print(f'INT4 + INT{ngram_bits} n-gram theoretical: {est["total_gib"]:.2f} GiB')

runtime = RuntimeConfig()
mem = estimate_runtime_memory(FlashNextConfig.tiny(), runtime, sessions=16, context=32768)
report = {'memory_estimate_gib': mem.as_gib(), 'devices': live_device_memory()}
(RESULTS / 'memory_report.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

In [ ]:
MODEL = os.environ.get('NEXUS_MODEL', 'Qwen/Qwen3.8-Flash-Next')
CHECKPOINT = os.environ.get('NEXUS_CHECKPOINT_ROOT')
benchmark = {'model': MODEL, 'checkpoint': CHECKPOINT, 'status': 'skipped'}

if not env_report['production_ok']:
    benchmark['reason'] = 'TPU v5e-8 not detected'
elif not CHECKPOINT and not os.environ.get('HF_TOKEN'):
    benchmark['reason'] = 'No checkpoint mounted and no HF_TOKEN — skipping full load'
else:
    try:
        t0 = time.time()
        cmd = [
            'vllm', 'serve', MODEL,
            '--tensor-parallel-size', '8',
            '--max-model-len', '4096',
            '--max-num-seqs', '2',
            '--disable-log-requests',
        ]
        if CHECKPOINT:
            cmd[2] = CHECKPOINT
        benchmark['compile_command'] = ' '.join(cmd)
        benchmark['status'] = 'manual_serve_required'
        benchmark['note'] = 'Run vllm serve in a separate cell; record compile vs steady-state separately.'
        benchmark['compile_seconds'] = None
    except Exception as exc:
        benchmark['status'] = 'failed'
        benchmark['error'] = str(exc)

(RESULTS / 'benchmark.json').write_text(json.dumps(benchmark, indent=2))
print(json.dumps(benchmark, indent=2))

## Context / concurrency sweep (TPU only)

When serving is available, sweep context lengths `4096 → 8192 → 16384 → 32768`, then increase concurrent sessions toward **16 × 32K**. Record OOM failures as results — do not fabricate throughput.